In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
import time

In [ ]:
from MMSNet import MMSDataset

dataset = MMSDataset(1)
dataloader = DataLoader(dataset , batch_size=1, shuffle=False, num_workers=1)

In [ ]:
from pynq import get_rails, DataRecorder

rails = get_rails()
recorder = DataRecorder(rails['12V'].power)

In [ ]:
from MMSNet import BaselineNet

model3 = BaselineNet()
model3.load_state_dict(torch.load('PreTrained_Weights/BaselineNet_336.pth'))

In [ ]:
def load_ip_input(UUT_ip, input_data, input_buffer):
    input_data_np = input_data.numpy()
    UUT_ip.write(0x10, input_buffer.physical_address)
    input_buffer[0] = input_data_np
    input_buffer.flush()
    

def get_ip_output(UUT_ip):
    output_1 = np.frombuffer(UUT_ip.read(0x20).to_bytes(4, 'little'), dtype=np.float32)
    output_2 = np.frombuffer(UUT_ip.read(0x24).to_bytes(4, 'little'), dtype=np.float32)
    output_3 = np.frombuffer(UUT_ip.read(0x28).to_bytes(4, 'little'), dtype=np.float32)
    output_4 = np.frombuffer(UUT_ip.read(0x2c).to_bytes(4, 'little'), dtype=np.float32)
    return np.concatenate((output_1, output_2, output_3, output_4), dtype=np.float32)

def start_ip(UUT_ip):
    UUT_ip.write(0x00, 0x01)
    while UUT_ip.read(0x00) & 0x4 == 0:
        continue

def run_ip(UUT_ip, input_data):
    load_ip_input(input_data)
    start_ip()
    return get_ip_output()

In [ ]:
from pynq import Overlay
from pynq import allocate

hls_output_dict = []
with recorder.record(0.05):
    ol = Overlay(f"hls_bitstream/BaselineNet.bit")
    recorder.mark()
    input_buffer = allocate(shape=(1, 1, 32,16,32), dtype=np.float32)
    weight_cv2 = model3.state_dict()["cv2.weight"]
    weight_fc1 = model3.state_dict()["fc1.weight"]
    tensor_cv2_weight_buffer = allocate(shape=weight_cv2.shape, dtype=np.float32)
    tensor_fc1_weight_buffer = allocate(shape=weight_fc1.shape, dtype=np.float32)
    tensor_cv2_weight_buffer[:] = weight_cv2.numpy()
    tensor_fc1_weight_buffer[:] = weight_fc1.numpy()
    UUT_ip = ol.entry_0
    UUT_ip.write(0x30, tensor_cv2_weight_buffer.physical_address)
    UUT_ip.write(0x3c, tensor_fc1_weight_buffer.physical_address)

    for data in dataloader:
        load_ip_input(UUT_ip, data, input_buffer[:])
        recorder.mark()
        start_ip(UUT_ip)
        recorder.mark()
        out_data = get_ip_output(UUT_ip)
        hls_output_dict.append(out_data)

In [ ]:
import matplotlib.pyplot as plt

# Get the timestamps of the marks
mark_indices = recorder.frame.index[recorder.frame['mark'] == True].tolist()

# Extract the power data
power_data = recorder.frame['12V_power']
time_data = recorder.frame.index

# Create the plot
plt.figure(figsize=(12, 6))
plt.plot(time_data, power_data, color='gray', label='Power Consumption')

# Color the plot based on the marks
if len(mark_indices) >= 1:
    plt.plot(time_data[:mark_indices[0]], power_data[:mark_indices[0]], color='red', label='Bitstream Download onto the FPGA')
if len(mark_indices) >= 2:
    plt.plot(time_data[mark_indices[0]:mark_indices[1]], power_data[mark_indices[0]:mark_indices[1]], color='blue', label='Writing/Reading NN accelerator MMIO registers')
if len(mark_indices) >= 3:
    plt.plot(time_data[mark_indices[1]:mark_indices[2]], power_data[mark_indices[1]:mark_indices[2]], color='orange', label='Executing accelerator')
if len(mark_indices) >= 3:
    plt.plot(time_data[mark_indices[2]:], power_data[mark_indices[2]:], color='blue', label='Writing/Reading NN accelerator MMIO registers')

# Add labels and title
plt.xlabel('Time')
plt.ylabel('Power (W)')
plt.title('Power Consumption During FPGA Execution')

# Add legend
plt.legend()

# Show the plot
plt.show()